<a href="https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Setup: DuckDB over the remote warehouse Parquet, HF token from Colab Secrets (never pasted in a cell).
%pip install -q duckdb

import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Enter your HF_TOKEN (not saved in this notebook): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Table paths — dims ship as one flat file, the daily fact ships partitioned by month.
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
FACT_MARCH  = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

MONTH_START = "DATE '2026-03-01'"
MONTH_MID   = "DATE '2026-03-15'"   # split point: first half vs second half of March
MONTH_END   = "DATE '2026-03-31'"

print("DuckDB ready. Secret registered. Paths set for the March 2026 partition (mid-panel month).")

DuckDB ready. Secret registered. Paths set for the March 2026 partition (mid-panel month).


In [3]:
# Schema peek — cheap (touches Parquet metadata, not data). Run this once per session
# to confirm real column names before trusting anything hardcoded below.
print("fact_content_daily_performance columns:")
display(con.sql(f"DESCRIBE SELECT * FROM {FACT_MARCH} LIMIT 0").df())

print("\ndim_content columns:")
display(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 0").df())

print("\ndim_clients columns:")
display(con.sql(f"DESCRIBE SELECT * FROM {DIM_CLIENTS} LIMIT 0").df())

fact_content_daily_performance columns:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



dim_content columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



dim_clients columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** One row (after the aggregation query in Section 3) = one content item's activity summary for March 2026 — a `client_hash_id` x `content_hash_id` x month grain, built by rolling up `fact_content_daily_performance`'s daily rows over the month. Before aggregation, the raw table's own grain is `report_date x client_hash_id x content_hash_id` (one row per page per day) — Section 3's grain query proves that first.

**Tables.** `fact_content_daily_performance` (primary, `month=2026-03` partition) for the daily search/analytics signals; `dim_content` (joined on `content_hash_id`) for static content metadata like `content_type` and `content_created_date`; `dim_clients` (referenced, not joined into the feature frame) only to sanity-check per-client `gsc_data_start` / `ga4_data_start` before trusting any zero.

**Time window.** March 1-31, 2026 — a mid-panel month, not the sealed `_sample` (which is June 2026, the final month, and the natural outcome window for any past-to-future label in this project). The decision point I'm framing around is "April 1, 2026: given everything known through March 31, which pages should a reviewer look at first?"

**Target / proxy.** `is_declining_march` — 1 when a page's GSC impressions in the second half of March (16th-31st) fell more than 20% versus the first half (1st-15th), else 0. This mirrors the exact `trend_direction == "down"` rule from the starter CSV's data dictionary, rebuilt here at real daily grain instead of trusting the pre-computed column.

**Deliberately excluded.** `fact_content_query_90d` — its fixed 90-day window does not align to a single calendar-month partition boundary (it always covers the most recent ~3 months of the whole snapshot), so joining it against a March-only contract would either pull in future months or leave stale trailing history. Out of scope for this month's contract; a query-mix feature set would need its own aligned contract.

In [4]:
# No query needed for this cell — Section 1 is verified by the grain/count queries in Section 3.
print("See Section 3 for the grain, count, and availability queries that back these claims.")

See Section 3 for the grain, count, and availability queries that back these claims.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context** (grouping/joining only, never a feature): `client_hash_id`, `content_hash_id`, `report_date` (used pre-aggregation, then collapsed away).

**Feature** (knowable before the April 1 decision point):
- `impressions_h1_march` — GSC impressions, March 1-15
- `clicks_h1_march` — GSC clicks, March 1-15
- `avg_position_march` — mean GSC position over the full month, excluding `gsc_avg_position = 0` ("no data") rows
- `sessions_h1_march` — GA4 sessions, March 1-15, only counted where `ga4_data_available IS TRUE`
- `content_age_days_mar31` — days between `dim_content.content_created_date` and March 31, 2026 (a fixed calendar fact, always knowable)
- `content_type` — static content metadata from `dim_content`

**Label / proxy** (the thing predicted, or its raw ingredients — never a feature): `impressions_h2_march` (GSC impressions, March 16-31) and the `is_declining_march` flag computed from it. Section 3's trap deliberately breaks this rule on purpose, then fixes it.

**Excluded**:
- `fact_content_query_90d` entirely — window doesn't align to the March partition (see Section 1).
- GA4 metrics for any client-month where `ga4_data_available` is not `TRUE` — excluded rather than zero-filled, because a client who hasn't started GA4 tracking yet looks identical to "zero engagement" unless the flag is checked explicitly (the three-valued `TRUE`/`FALSE`/`NULL` gotcha from the data skill).
- `dim_clients.gsc_data_start` / `ga4_data_start` — used only as a lookup to sanity-check coverage, never joined into the feature frame or fed to a model.

In [5]:
# No query needed for this cell — the field classification is verified alongside the
# grain/count/availability queries and the feature-frame build in Section 3.
print("See Section 3.")

See Section 3.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Fact 1 — grain.** Prove one row of the raw daily fact table really is one page-day (before any aggregation).

In [6]:
# Fact 1 — grain check on the raw March partition (should return ZERO rows).
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows with a duplicated (report_date, client_hash_id, content_hash_id) key: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with a duplicated (report_date, client_hash_id, content_hash_id) key: 0


,report_date,client_hash_id,content_hash_id,n


**Fact 2 — row count and date span.** Confirm the March partition holds what it claims to.

In [7]:
# Fact 2 — counts and date span for the March 2026 partition.
counts = con.sql(f"""
    SELECT
        COUNT(*)                       AS total_rows,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        MIN(report_date)               AS min_date,
        MAX(report_date)               AS max_date
    FROM {FACT_MARCH}
""").df()

counts

,total_rows,distinct_clients,distinct_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


**Fact 3 — availability.** Filter with `IS TRUE` (not just non-null) and show how many rows survive.

In [8]:
# Fact 3 — GA4 availability. The flag is three-valued (TRUE / FALSE / NULL), so `IS TRUE`
# is the only safe filter — `= TRUE` and `NOT ... = FALSE` both mishandle NULL rows.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {FACT_MARCH}
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


**Five features.** Build the page-month feature frame for March 2026, one row per content item, and give each feature a one-line "knowable at the decision moment because..." justification (already written in Section 2 — restated inline in the query comments below).

In [9]:
# Aggregate the daily fact table up to one row per (client, content) for March 2026,
# then join static content metadata. This IS the feature frame.
feature_frame = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            -- 1) impressions_h1_march: known because GSC impressions for days 1-15 are
            --    already recorded by the time anyone looks at this on April 1.
            SUM(CASE WHEN report_date < {MONTH_MID} THEN gsc_impressions ELSE 0 END) AS impressions_h1_march,
            -- 2) clicks_h1_march: same reasoning — recorded GSC clicks, first half of March.
            SUM(CASE WHEN report_date < {MONTH_MID} THEN gsc_clicks ELSE 0 END)      AS clicks_h1_march,
            -- 3) avg_position_march: position is measured daily as pages get ranked;
            --    fully known by March 31. gsc_avg_position = 0 means "no data", excluded.
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)            AS avg_position_march,
            -- 4) sessions_h1_march: GA4 analytics land same-day/next-day, so known before
            --    April 1 — but only trusted where ga4_data_available IS TRUE.
            SUM(CASE WHEN report_date < {MONTH_MID} AND ga4_data_available IS TRUE
                     THEN ga4_sessions ELSE 0 END)                                    AS sessions_h1_march,
            -- LABEL INGREDIENT — second half of March. Kept here on purpose for Section 3's
            -- trap below, then dropped before any honest feature frame is used.
            SUM(CASE WHEN report_date >= {MONTH_MID} THEN gsc_impressions ELSE 0 END) AS impressions_h2_march
        FROM {FACT_MARCH}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.*,
        -- 5) content_age_days_mar31: a fixed calendar fact from a creation date already in
        --    the past — always knowable at any future decision point.
        DATE_DIFF('day', c.content_created_date, {MONTH_END}) AS content_age_days_mar31,
        c.content_type,
        -- proxy label: same >20% drop rule as the starter CSV's trend_direction == "down",
        -- rebuilt from raw daily data instead of trusted from a pre-computed column.
        CASE
            WHEN d.impressions_h1_march > 0
                 AND (d.impressions_h2_march - d.impressions_h1_march) / d.impressions_h1_march < -0.20
            THEN 1 ELSE 0
        END AS is_declining_march
    FROM daily_agg d
    JOIN {DIM_CONTENT} c ON d.content_hash_id = c.content_hash_id
    WHERE d.impressions_h1_march > 0   -- need a first-half baseline to compute the trend at all
""").df()

print(f"Feature frame: {len(feature_frame):,} content items x {feature_frame.shape[1]} columns")
print(f"Proxy label rate this month: {feature_frame['is_declining_march'].mean():.1%}")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 150,444 content items x 10 columns
Proxy label rate this month: 27.6%


,client_hash_id,content_hash_id,impressions_h1_march,clicks_h1_march,avg_position_march,sessions_h1_march,impressions_h2_march,content_age_days_mar31,content_type,is_declining_march
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4003.0,6.0,7.209549,0.0,2520.0,396,keyword article,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,231.0,0.0,3.307255,0.0,222.0,396,keyword article,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3549.0,3.0,6.724039,0.0,2081.0,396,keyword article,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2395.0,8.0,7.244844,0.0,2549.0,396,keyword article,0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,13.0,0.0,23.314103,0.0,29.0,396,keyword article,0


**The trap.** Add the label's own raw ingredient (`impressions_h2_march`) as if it were a legitimate feature, watch a quick ranking score jump toward perfect, then delete it and keep the honest number.

In [10]:
# THE TRAP — on purpose.
# Rank pages by ascending impressions_h2_march (lowest second-half impressions first) and
# score against the proxy label with Precision@50, same metric chosen back in ML-03.
def precision_at_k(frame, sort_col, ascending, label_col="is_declining_march", k=50):
    ranked = frame.sort_values(sort_col, ascending=ascending)
    top_k = ranked.head(k)
    return top_k[label_col].mean(), int(top_k[label_col].sum())

leak_precision, leak_matches = precision_at_k(feature_frame, "impressions_h2_march", ascending=True)
print("LEAKED ranking (sorted by impressions_h2_march, the label's own raw ingredient):")
print(f"  Precision@50 = {leak_precision:.2f}  ->  {leak_matches} of 50 top-ranked pages 'match'")
print("  This is fake. impressions_h2_march is literally half the arithmetic that BUILDS the label.")
print()

# Now delete the leaked column and re-check with an honest, legitimate feature instead.
honest_frame = feature_frame.drop(columns=["impressions_h2_march"])
honest_precision, honest_matches = precision_at_k(honest_frame, "avg_position_march", ascending=False)

print("HONEST ranking (sorted by avg_position_march, a legitimate first-half/full-month feature):")
print(f"  Precision@50 = {honest_precision:.2f}  ->  {honest_matches} of 50 top-ranked pages match")
print()
print(f"Leaked score {leak_precision:.2f} vs honest score {honest_precision:.2f} — the gap IS the leak.")
print("Keeping only the honest number and the honest_frame (no h2 columns) going forward.")

LEAKED ranking (sorted by impressions_h2_march, the label's own raw ingredient):
  Precision@50 = 1.00  ->  50 of 50 top-ranked pages 'match'
  This is fake. impressions_h2_march is literally half the arithmetic that BUILDS the label.

HONEST ranking (sorted by avg_position_march, a legitimate first-half/full-month feature):
  Precision@50 = 0.76  ->  38 of 50 top-ranked pages match

Leaked score 1.00 vs honest score 0.76 — the gap IS the leak.
Keeping only the honest number and the honest_frame (no h2 columns) going forward.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: unbalanced panel coverage inside March 2026.** Not every client had tracking running for the full month. Per-client history depth varies wildly (`dim_clients.gsc_data_start` / `ga4_data_start`), so a client whose GSC tracking only started mid-March will show a `impressions_h1_march` near zero for reasons that have nothing to do with content quality — and my `impressions_h1_march > 0` filter silently drops those rows rather than flagging them, so this month's feature frame is systematically missing clients who onboarded partway through March 2026. That's a real gap this data cannot fill on its own: a page with a genuinely thin first half looks identical to a page whose *tracking* only started on the 20th. Before trusting `is_declining_march` for any single client, I'd need to check that client's `gsc_data_start` against the March window first — this contract does not yet do that check.

Two smaller limits worth naming: (1) this month's GA4 features only cover client-months where `ga4_data_available IS TRUE` — clients still on GSC-only tracking simply have no `sessions_h1_march` signal, not a zero one; (2) a single mid-panel month cannot show seasonality or persistence — a real decline (Section 7 of the lane guide) needs the drop to hold across more than one 30-day comparison, which this one-month contract does not test.

In [11]:
# Quick evidence for the named limitation: how many distinct clients appear in the March
# partition at all, versus how many clients dim_clients says exist overall.
coverage = con.sql(f"""
    SELECT
        (SELECT COUNT(DISTINCT client_hash_id) FROM {FACT_MARCH}) AS clients_active_in_march,
        (SELECT COUNT(*) FROM {DIM_CLIENTS}) AS clients_total_in_warehouse
""").df()

coverage

,clients_active_in_march,clients_total_in_warehouse
0,55,104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.